**This notebook is an exercise in the [Machine Learning Explainability](https://www.kaggle.com/learn/machine-learning-explainability) course.  You can reference the tutorial at [this link](https://www.kaggle.com/dansbecker/permutation-importance).**

---


## Intro

You will think about and calculate permutation importance with a sample of data from the [Taxi Fare Prediction](https://www.kaggle.com/c/new-york-city-taxi-fare-prediction) competition.

We won't focus on data exploration or model building for now. You can just run the cell below to 
- Load the data
- Divide the data into training and validation
- Build a model that predicts taxi fares
- Print a few rows for you to review

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

data = pd.read_csv('train.csv', nrows=50000)

# 2. Check columns to prevent KeyErrors
if 'fare_amount' in data.columns:
    # 3. Filter using standard indexing (safer than .query in some environments)
    data = data[(data.pickup_latitude > 40.7) & (data.pickup_latitude < 40.8) &
                (data.dropoff_latitude > 40.7) & (data.dropoff_latitude < 40.8) &
                (data.pickup_longitude > -74) & (data.pickup_longitude < -73.9) &
                (data.dropoff_longitude > -74) & (data.dropoff_longitude < -73.9) &
                (data.fare_amount > 0)]

    y = data.fare_amount
    base_features = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']
    X = data[base_features]

    train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

    first_model = RandomForestRegressor(n_estimators=50, random_state=1).fit(train_X, train_y)
    print("Model trained successfully!")
else:
    print(f"Error: 'fare_amount' not found. Available columns are: {list(data.columns)}")

Model trained successfully!


The following two cells may also be useful to understand the values in the training data:

In [7]:
train_X.describe()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,3188.000000,3188.000000,3188.000000,3188.000000,3188.000000
mean,-73.950864,40.749514,-73.950557,40.749635,2.974279
std,0.026757,0.027364,0.026926,0.027051,1.400826
min,-73.999960,40.700003,-73.999991,40.700019,1.000000
25%,-73.973731,40.726027,-73.973417,40.726650,2.000000
50%,-73.951471,40.750038,-73.951331,40.750509,3.000000
75%,-73.928220,40.772231,-73.927811,40.772350,4.000000
max,-73.900110,40.799972,-73.900008,40.799952,5.000000


In [8]:
train_y.describe()

count    3188.000000
mean        3.786588
std         1.041208
min         2.500000
25%         2.864122
50%         3.669795
75%         4.475591
max         8.370755
Name: fare_amount, dtype: float64

## Question 1

The first model uses the following features
- pickup_longitude
- pickup_latitude
- dropoff_longitude
- dropoff_latitude
- passenger_count

## Question 2

Create a `PermutationImportance` object called `perm` to show the importances from `first_model`.  Fit it with the appropriate data and show the weights.

For your convenience, the code from the tutorial has been copied into a comment in this code cell.

In [10]:
%pip install eli5
import eli5
from eli5.sklearn import PermutationImportance

# Make a small change to the code below to use in this problem.
perm = PermutationImportance(first_model, random_state=1).fit(val_X, val_y)


# uncomment the following line to visualize your results
eli5.show_weights(perm, feature_names = val_X.columns.tolist())

  Obtaining dependency information for eli5 from https://files.pythonhosted.org/packages/68/a0/23e75b17140708414fbe01b3bdc43ed3b8ed43933fe8f18466623c1fc251/eli5-0.16.0-py2.py3-none-any.whl.metadata
  Obtaining dependency information for attrs>17.1.0 from https://files.pythonhosted.org/packages/64/b4/17d4b0b2a2dc85a6df63d1157e028ed19f90d4cd97c36717afef2bc2f395/attrs-26.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for jinja2>=3.0.0 from https://files.pythonhosted.org/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e594bb582bc3b64ea099db13e56/jinja2-3.1.6-py3-none-any.whl.metadata
  Obtaining dependency information for graphviz from https://files.pythonhosted.org/packages/91/4c/e0ce1ef95d4000ebc1c11801f9b944fa5910ecc15b5e351865763d8657f8/graphviz-0.21-py3-none-any.whl.metadata
  Obtaining dependency information for tabulate>=0.7.7 from https://files.pythonhosted.org/packages/99/55/db07de81b5c630da5cbf5c7df646580ca26dfaefa593667fc6f2fe016d2e/tabulate-0.10.0-py3-none-any

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Weight,Feature
1.1716 ± 0.0718,pickup_longitude
1.1490 ± 0.0936,dropoff_longitude
0.5621 ± 0.0587,dropoff_latitude
0.5581 ± 0.0415,pickup_latitude
-0.0014 ± 0.0053,passenger_count


## Question 4

Without detailed knowledge of New York City, it's difficult to rule out most hypotheses about why latitude features matter more than longitude.

A good next step is to disentangle the effect of being in certain parts of the city from the effect of total distance traveled.  

The code below creates new features for longitudinal and latitudinal distance. It then builds a model that adds these new features to those you already had.

Fill in two lines of code to calculate and show the importance weights with this new set of features. As usual, you can uncomment lines below to check your code, see a hint or get the solution.

In [11]:
# create new features
data['abs_lon_change'] = abs(data.dropoff_longitude - data.pickup_longitude)
data['abs_lat_change'] = abs(data.dropoff_latitude - data.pickup_latitude)

features_2  = ['pickup_longitude',
               'pickup_latitude',
               'dropoff_longitude',
               'dropoff_latitude',
               'abs_lat_change',
               'abs_lon_change']

X = data[features_2]
new_train_X, new_val_X, new_train_y, new_val_y = train_test_split(X, y, random_state=1)
second_model = RandomForestRegressor(n_estimators=30, random_state=1).fit(new_train_X, new_train_y)

# Create a PermutationImportance object on second_model and fit it to new_val_X and new_val_y
# Use a random_state of 1 for reproducible results that match the expected solution.
perm2 = PermutationImportance(second_model , random_state=1).fit(new_val_X, new_val_y)
eli5.show_weights(perm2, feature_names = new_val_X.columns.tolist())


Weight,Feature
0.2954 ± 0.0375,abs_lon_change
0.2583 ± 0.0270,abs_lat_change
0.0014 ± 0.0067,dropoff_latitude
0.0001 ± 0.0091,pickup_longitude
-0.0039 ± 0.0195,pickup_latitude
-0.0069 ± 0.0120,dropoff_longitude
